In [18]:
import cv2
import numpy as np

def preprocess_cxr(image_path, target_size=(224, 224)):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None

    # CLAHE (paper method)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    enhanced = clahe.apply(img)

    # Resize
    resized = cv2.resize(enhanced, target_size)

    # Convert to 3-channel
    return np.stack([resized]*3, axis=-1)

In [20]:
import os
from torchvision import transforms, datasets
from torch.utils.data import DataLoader

data_dir = "Final_Data"

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.1, contrast=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

train_dataset = datasets.ImageFolder(os.path.join(data_dir, "train"), transform=train_transform)
val_dataset = datasets.ImageFolder(os.path.join(data_dir, "val"), transform=val_transform)
test_dataset = datasets.ImageFolder(os.path.join(data_dir, "test"), transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32)
test_loader = DataLoader(test_dataset, batch_size=32)

print(train_dataset.classes)

['Covid-19', 'Normal', 'Pneumonia-Bacterial', 'Pneumonia-Viral']


In [5]:
import os
import cv2
import numpy as np
from tqdm import tqdm

# ==============================
# 📁 PATHS
# ==============================
input_dir = "/Users/kpvarma/PycharmProjects/Pneumonia_Research/Final_Data"
output_dir = "/Users/kpvarma/PycharmProjects/Pneumonia_Research/Final_Data_CLAHE"

splits = ["train", "val", "test"]

# ==============================
# 🔥 CREATE CLAHE OBJECT
# ==============================
clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))


# ==============================
# 🔁 PROCESS DATASET
# ==============================
for split in splits:

    split_path = os.path.join(input_dir, split)

    for cls in os.listdir(split_path):

        input_class_dir = os.path.join(split_path, cls)

        # 🔥 FIX: skip non-directories
        if not os.path.isdir(input_class_dir):
            continue

        output_class_dir = os.path.join(output_dir, split, cls)
        os.makedirs(output_class_dir, exist_ok=True)

        images = [f for f in os.listdir(input_class_dir)
                  if f.lower().endswith((".png", ".jpg", ".jpeg"))]

        print(f"\nProcessing {split}/{cls} ({len(images)} images)")

        for img_name in tqdm(images):

            img_path = os.path.join(input_class_dir, img_name)

            img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)

            if img is None:
                continue

            enhanced = clahe.apply(img)
            enhanced = np.stack([enhanced]*3, axis=-1)

            save_path = os.path.join(output_class_dir, img_name)
            cv2.imwrite(save_path, enhanced)
print("\n✅ CLAHE dataset creation complete!")


Processing train/Pneumonia-Bacterial (1887 images)


100%|██████████| 1887/1887 [00:01<00:00, 1137.26it/s]



Processing train/Pneumonia-Viral (1887 images)


100%|██████████| 1887/1887 [00:01<00:00, 1211.53it/s]



Processing train/Covid-19 (1887 images)


...


Processing test/Covid-19 (405 images)


100%|██████████| 405/405 [00:00<00:00, 1219.11it/s]



Processing test/Normal (405 images)


100%|██████████| 405/405 [00:00<00:00, 1086.98it/s]


✅ CLAHE dataset creation complete!
